# Notebook 03 — Sobol Sensitivity Analysis

Use SALib to perform Sobol sensitivity analysis with 8 input parameters.
Saltelli sampling N=512 giving 5120 evaluations.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from SALib.sample import saltelli
from SALib.analyze import sobol
from tqdm import tqdm

from src.simulation import FleetSimulation

%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120

print('Ready.')

## Define Problem

In [ ]:
problem = {
    'num_vars': 8,
    'names': ['engine_eta', 'avionics_eta', 'hydraulics_eta',
              'engine_reorder_point', 'engine_lead_time_mean',
              'o_level_technicians_day', 'i_level_technicians_day',
              'sortie_interval_hours'],
    'bounds': [
        [600, 1000],    # engine_eta
        [400, 800],     # avionics_eta
        [800, 1600],    # hydraulics_eta
        [1, 4],         # engine_reorder_point
        [7, 21],        # engine_lead_time_mean (days)
        [4, 12],        # o_level_technicians_day
        [2, 6],         # i_level_technicians_day
        [36, 60],       # sortie_interval_hours
    ]
}

# Generate Saltelli samples
N = 512
param_values = saltelli.sample(problem, N)
print(f'Total evaluations: {len(param_values)} (N={N}, k={problem["num_vars"]})')

## Run Evaluations

In [ ]:
Y = np.zeros(len(param_values))

for i, X in enumerate(tqdm(param_values, desc='Sobol evaluations')):
    subsystem_params = {
        'engine': {'eta': X[0]},
        'avionics': {'eta': X[1]},
        'hydraulics': {'eta': X[2]},
    }
    inventory_params = {
        'engine': {
            'reorder_point': int(round(X[3])),
            'lead_time_mean_days': X[4],
        }
    }
    
    sim = FleetSimulation(
        fleet_size=12,
        o_level_techs_day=int(round(X[5])),
        i_level_techs_day=int(round(X[6])),
        sortie_interval_hours=X[7],
        subsystem_params=subsystem_params,
        inventory_params=inventory_params,
    )
    
    result = sim.run(simulation_days=90, random_seed=42)
    Y[i] = result['mcr']

print(f'\nMCR range: [{Y.min()*100:.1f}%, {Y.max()*100:.1f}%]')
print(f'MCR mean: {Y.mean()*100:.1f}%')

## Sobol Analysis

In [ ]:
Si = sobol.analyze(problem, Y, print_to_console=True)

## 1. Bar Chart of S1 and ST

In [ ]:
names = problem['names']
S1 = Si['S1']
ST = Si['ST']
S1_conf = Si['S1_conf']
ST_conf = Si['ST_conf']

# Sort by ST
sort_idx = np.argsort(ST)
sorted_names = [names[i] for i in sort_idx]
sorted_S1 = S1[sort_idx]
sorted_ST = ST[sort_idx]
sorted_S1_conf = S1_conf[sort_idx]
sorted_ST_conf = ST_conf[sort_idx]

y_pos = np.arange(len(names))

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(y_pos - 0.15, sorted_S1, 0.3, xerr=sorted_S1_conf,
        color='#3498db', alpha=0.8, label='S1 (First-order)')
ax.barh(y_pos + 0.15, sorted_ST, 0.3, xerr=sorted_ST_conf,
        color='#e74c3c', alpha=0.8, label='ST (Total-order)')

ax.set_yticks(y_pos)
ax.set_yticklabels(sorted_names)
ax.set_xlabel('Sensitivity Index', fontsize=12)
ax.set_title('Sobol Sensitivity Indices (S1 and ST) with 95% Bootstrap CIs',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 2. Interpretation

In [ ]:
print('INTERPRETATION')
print('=' * 60)
print('\nDominant parameters (ST > 0.10):')
for name, st_val in zip(names, ST):
    if st_val > 0.10:
        print(f'  • {name}: ST = {st_val:.3f}')

print('\nNon-influential parameters (ST < 0.05):')
for name, st_val in zip(names, ST):
    if st_val < 0.05:
        print(f'  • {name}: ST = {st_val:.3f}')